# Batch Processing with NEUIToolkit

This notebook demonstrates how to efficiently process multiple documents in parallel using NEUIToolkit.

## What You'll Learn

- Process multiple documents in parallel
- Optimize worker configuration
- Handle errors gracefully
- Monitor progress and performance
- Aggregate results across documents
- Cost optimization strategies

## Prerequisites

- Multiple documents to process (PDF, DOCX, TXT, etc.)
- LLM API key configured
- Understanding of concurrent processing

## Setup

In [ ]:
import sys
import os
from pathlib import Path
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any
from tqdm.notebook import tqdm

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import NEUIToolkit modules
from backend.orchestrator import (
    read_corpus,
    run_entity_pass,
    run_relationship_pass,
    run_rule_pass,
    run_ontology_pass
)
from backend.quality_assurance import QualityAssurance
from llm.llm_utils import get_provider_stats

print("✅ Modules loaded successfully!")

## Step 1: Prepare Sample Documents

Create multiple sample documents for batch processing.

In [ ]:
# Create sample documents directory
sample_dir = Path("./sample_batch_docs")
sample_dir.mkdir(exist_ok=True)

# Sample texts on different topics
samples = {
    "biology.txt": """
    Cells are the basic unit of life. The cell membrane controls what enters and exits.
    Mitochondria produce ATP through cellular respiration. The nucleus contains DNA.
    If a cell divides, then two daughter cells are formed.
    """,
    
    "physics.txt": """
    Energy cannot be created or destroyed, only transformed. Force equals mass times acceleration.
    Light travels at 299,792 kilometers per second in vacuum. Gravity attracts all masses.
    If an object is in motion, then it will remain in motion unless acted upon by a force.
    """,
    
    "chemistry.txt": """
    Atoms consist of protons, neutrons, and electrons. Chemical bonds form between atoms.
    The periodic table organizes elements by atomic number. Water is composed of hydrogen and oxygen.
    If atoms share electrons, then a covalent bond forms.
    """,
    
    "computer_science.txt": """
    Algorithms are step-by-step procedures for solving problems. Data structures organize information.
    Binary represents data using zeros and ones. Recursion allows functions to call themselves.
    If input size doubles, then O(n²) algorithms take four times longer.
    """,
    
    "psychology.txt": """
    Memory stores and retrieves information. Cognition involves mental processes like thinking.
    Emotions influence decision-making. The brain processes sensory information.
    If practice occurs regularly, then skills improve through reinforcement.
    """
}

# Write sample files
for filename, content in samples.items():
    (sample_dir / filename).write_text(content.strip())

print(f"✅ Created {len(samples)} sample documents in {sample_dir}/")
for filename in samples.keys():
    print(f"   • {filename}")

## Step 2: Single Document Processing (Baseline)

First, let's process a single document to establish a baseline.

In [ ]:
def process_single_document(file_path: Path, qa_system: QualityAssurance = None) -> Dict[str, Any]:
    """Process a single document and return results.
    
    Args:
        file_path: Path to document
        qa_system: Quality assurance system (optional)
    
    Returns:
        Dictionary with extraction results and metadata
    """
    start_time = time.time()
    
    try:
        # Read document
        corpus = read_corpus(file_path)
        
        # Extract knowledge
        entities = run_entity_pass(corpus)
        relationships = run_relationship_pass(entities, corpus)
        rules = run_rule_pass(corpus)
        
        # Apply QA if available
        quality_score = None
        if qa_system:
            entities, entity_metrics = qa_system.filter_low_quality(entities, 'entity')
            relationships, rel_metrics = qa_system.filter_low_quality(relationships, 'relationship')
            rules, rule_metrics = qa_system.filter_low_quality(rules, 'rule')
            quality_report = qa_system.generate_quality_report(entity_metrics, rel_metrics, rule_metrics)
            quality_score = quality_report['overall_quality_score']
        
        processing_time = time.time() - start_time
        
        return {
            "filename": file_path.name,
            "success": True,
            "entities": entities,
            "relationships": relationships,
            "rules": rules,
            "num_entities": len(entities),
            "num_relationships": len(relationships),
            "num_rules": len(rules),
            "quality_score": quality_score,
            "processing_time": processing_time
        }
    
    except Exception as e:
        return {
            "filename": file_path.name,
            "success": False,
            "error": str(e),
            "processing_time": time.time() - start_time
        }

# Test with single document
test_file = list(sample_dir.glob("*.txt"))[0]
qa = QualityAssurance(min_confidence=0.5)

result = process_single_document(test_file, qa)

print(f"✅ Processed {result['filename']} in {result['processing_time']:.2f}s")
print(f"   Entities: {result['num_entities']}")
print(f"   Relationships: {result['num_relationships']}")
print(f"   Rules: {result['num_rules']}")
print(f"   Quality: {result['quality_score']:.3f}")

## Step 3: Sequential Processing (No Parallelism)

Process all documents one at a time.

In [ ]:
files = sorted(sample_dir.glob("*.txt"))
qa = QualityAssurance(min_confidence=0.5)

print(f"Processing {len(files)} documents sequentially...\n")

sequential_start = time.time()
sequential_results = []

for file in tqdm(files, desc="Processing"):
    qa.reset()  # Reset QA for each document
    result = process_single_document(file, qa)
    sequential_results.append(result)

sequential_total_time = time.time() - sequential_start

print(f"\n✅ Sequential processing complete!")
print(f"   Total time: {sequential_total_time:.2f}s")
print(f"   Average time per document: {sequential_total_time/len(files):.2f}s")
print(f"   Documents per minute: {len(files)/(sequential_total_time/60):.1f}")

## Step 4: Parallel Processing

Process documents in parallel using ThreadPoolExecutor.

In [ ]:
def batch_process_documents(
    file_paths: List[Path],
    max_workers: int = 4,
    use_qa: bool = True
) -> List[Dict[str, Any]]:
    """Process multiple documents in parallel.
    
    Args:
        file_paths: List of paths to documents
        max_workers: Number of parallel workers
        use_qa: Whether to use quality assurance
    
    Returns:
        List of results dictionaries
    """
    results = []
    
    # Create QA systems per worker (thread-safe)
    qa_systems = {i: QualityAssurance(min_confidence=0.5) for i in range(max_workers)} if use_qa else {}
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_file = {
            executor.submit(
                process_single_document,
                file_path,
                qa_systems.get(i % max_workers) if use_qa else None
            ): file_path
            for i, file_path in enumerate(file_paths)
        }
        
        # Collect results as they complete
        for future in tqdm(as_completed(future_to_file), total=len(future_to_file), desc="Processing"):
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                file_path = future_to_file[future]
                results.append({
                    "filename": file_path.name,
                    "success": False,
                    "error": str(e)
                })
    
    return results

# Test with different worker counts
worker_configs = [1, 2, 4]
parallel_results = {}

for num_workers in worker_configs:
    print(f"\nTesting with {num_workers} worker(s)...")
    
    start_time = time.time()
    results = batch_process_documents(files, max_workers=num_workers, use_qa=True)
    total_time = time.time() - start_time
    
    parallel_results[num_workers] = {
        "results": results,
        "total_time": total_time,
        "avg_time": total_time / len(files),
        "docs_per_minute": len(files) / (total_time / 60)
    }
    
    print(f"   Total time: {total_time:.2f}s")
    print(f"   Average time: {total_time/len(files):.2f}s/doc")
    print(f"   Throughput: {len(files)/(total_time/60):.1f} docs/min")

## Step 5: Performance Comparison

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Prepare data for visualization
comparison_data = {
    "Sequential (1 worker)": sequential_total_time,
}
comparison_data.update({
    f"{w} workers": parallel_results[w]["total_time"]
    for w in worker_configs
})

# Plot comparison
plt.figure(figsize=(10, 6))
configs = list(comparison_data.keys())
times = list(comparison_data.values())

bars = plt.bar(configs, times, color=['red'] + ['skyblue'] * len(worker_configs))
plt.ylabel('Total Time (seconds)')
plt.title('Processing Time Comparison: Sequential vs Parallel')
plt.xticks(rotation=15, ha='right')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}s',
             ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Calculate speedup
print("\nSpeedup Analysis:")
baseline = sequential_total_time
for workers, data in parallel_results.items():
    speedup = baseline / data["total_time"]
    efficiency = speedup / workers * 100
    print(f"  {workers} workers: {speedup:.2f}x speedup ({efficiency:.1f}% efficiency)")

## Step 6: Aggregate Results Analysis

In [ ]:
# Use results from best performing configuration
best_workers = max(parallel_results.keys(), key=lambda w: parallel_results[w]["docs_per_minute"])
results = parallel_results[best_workers]["results"]

# Aggregate statistics
successful = [r for r in results if r.get("success", False)]
failed = [r for r in results if not r.get("success", False)]

print(f"Batch Processing Summary ({best_workers} workers):")
print("=" * 60)
print(f"Total documents: {len(results)}")
print(f"Successful: {len(successful)}")
print(f"Failed: {len(failed)}")
print()

if successful:
    total_entities = sum(r["num_entities"] for r in successful)
    total_relationships = sum(r["num_relationships"] for r in successful)
    total_rules = sum(r["num_rules"] for r in successful)
    avg_quality = sum(r["quality_score"] for r in successful) / len(successful)
    
    print("Extraction Statistics:")
    print(f"  Total entities: {total_entities}")
    print(f"  Total relationships: {total_relationships}")
    print(f"  Total rules: {total_rules}")
    print(f"  Average quality: {avg_quality:.3f}")
    print()
    
    print("Per-Document Averages:")
    print(f"  Entities: {total_entities/len(successful):.1f}")
    print(f"  Relationships: {total_relationships/len(successful):.1f}")
    print(f"  Rules: {total_rules/len(successful):.1f}")

# Show individual results
print("\n" + "=" * 60)
print("Individual Document Results:")
print("=" * 60)

for r in successful:
    print(f"\n{r['filename']}:")
    print(f"  Entities: {r['num_entities']}")
    print(f"  Relationships: {r['num_relationships']}")
    print(f"  Rules: {r['num_rules']}")
    print(f"  Quality: {r['quality_score']:.3f}")
    print(f"  Time: {r['processing_time']:.2f}s")

## Step 7: Cost Analysis

Analyze LLM API costs for batch processing.

In [ ]:
try:
    # Get LLM provider statistics
    llm_stats = get_provider_stats()
    
    print("LLM Cost Analysis:")
    print("=" * 60)
    print(f"Provider: {llm_stats.get('primary_provider', 'N/A')}")
    print(f"Total API calls: {llm_stats.get('total_calls', 0)}")
    print(f"Total cost: ${llm_stats.get('total_cost', 0):.4f}")
    print(f"Cost per document: ${llm_stats.get('total_cost', 0)/len(files):.4f}")
    print(f"Cost per entity: ${llm_stats.get('total_cost', 0)/total_entities:.4f}")
    print()
    
    # Extrapolate for larger batches
    batch_sizes = [10, 50, 100, 500, 1000]
    cost_per_doc = llm_stats.get('total_cost', 0) / len(files)
    
    print("Cost Projections for Larger Batches:")
    print("-" * 60)
    for size in batch_sizes:
        projected_cost = cost_per_doc * size
        projected_time = (parallel_results[best_workers]["total_time"] / len(files)) * size
        print(f"  {size:4d} documents: ${projected_cost:7.2f} (~{projected_time/60:.1f} min)")
    
except Exception as e:
    print(f"Could not retrieve LLM statistics: {e}")

## Step 8: Error Handling & Recovery

Demonstrate robust error handling for production use.

In [ ]:
def robust_batch_process(
    file_paths: List[Path],
    max_workers: int = 4,
    max_retries: int = 3,
    output_dir: Path = None
) -> Dict[str, Any]:
    """Process documents with error handling and recovery.
    
    Args:
        file_paths: Documents to process
        max_workers: Number of parallel workers
        max_retries: Maximum retry attempts for failed documents
        output_dir: Directory to save results (optional)
    
    Returns:
        Summary dictionary with results and statistics
    """
    results = []
    failed_files = []
    
    # Initial processing
    print(f"Processing {len(file_paths)} documents with {max_workers} workers...")
    initial_results = batch_process_documents(file_paths, max_workers=max_workers)
    
    # Separate successes and failures
    for result in initial_results:
        if result.get("success", False):
            results.append(result)
        else:
            failed_files.append(result["filename"])
    
    # Retry failed files
    retry_count = 0
    while failed_files and retry_count < max_retries:
        retry_count += 1
        print(f"\nRetry {retry_count}/{max_retries} for {len(failed_files)} failed documents...")
        
        retry_paths = [f for f in file_paths if f.name in failed_files]
        retry_results = batch_process_documents(retry_paths, max_workers=max_workers)
        
        failed_files = []
        for result in retry_results:
            if result.get("success", False):
                results.append(result)
            else:
                failed_files.append(result["filename"])
    
    # Save results if output_dir provided
    if output_dir:
        output_dir.mkdir(exist_ok=True)
        for result in results:
            if result.get("success"):
                filename = Path(result["filename"]).stem
                
                # Save entities
                with open(output_dir / f"{filename}_entities.json", "w") as f:
                    json.dump(result["entities"], f, indent=2)
                
                # Save relationships
                with open(output_dir / f"{filename}_relationships.json", "w") as f:
                    json.dump(result["relationships"], f, indent=2)
                
                # Save rules
                with open(output_dir / f"{filename}_rules.json", "w") as f:
                    json.dump(result["rules"], f, indent=2)
    
    # Compile summary
    summary = {
        "total_documents": len(file_paths),
        "successful": len(results),
        "failed": len(failed_files),
        "failed_files": failed_files,
        "retry_count": retry_count,
        "results": results
    }
    
    return summary

# Test robust processing
output_dir = Path("./batch_results")
summary = robust_batch_process(files, max_workers=4, max_retries=3, output_dir=output_dir)

print("\n" + "=" * 60)
print("Robust Batch Processing Summary:")
print("=" * 60)
print(f"Total documents: {summary['total_documents']}")
print(f"Successful: {summary['successful']}")
print(f"Failed: {summary['failed']}")
print(f"Retry attempts: {summary['retry_count']}")

if summary['failed_files']:
    print(f"\nFailed files:")
    for f in summary['failed_files']:
        print(f"  - {f}")

if output_dir:
    print(f"\n✅ Results saved to: {output_dir.absolute()}")

## Best Practices for Batch Processing

### 1. Worker Configuration
- **CPU-bound tasks**: Use `max_workers = CPU count`
- **I/O-bound tasks** (LLM calls): Use `max_workers = 2-4x CPU count`
- **API rate limits**: Adjust workers to stay under limits
- **Memory constraints**: Reduce workers if running out of memory

### 2. Error Handling
- Always implement retry logic
- Log failures with details
- Save partial results
- Implement timeout mechanisms

### 3. Cost Optimization
- Use cheaper models for simple extractions
- Cache results to avoid reprocessing
- Batch LLM requests when possible
- Monitor and set budget limits

### 4. Performance Monitoring
- Track processing time per document
- Monitor memory usage
- Log API call statistics
- Set up alerts for failures

### 5. Quality Control
- Always use QA layer in production
- Set appropriate confidence thresholds
- Review low-quality extractions
- Implement human-in-the-loop for critical data

## Production Checklist

Before deploying batch processing to production:

- [ ] Test with representative document sample
- [ ] Configure appropriate worker count
- [ ] Implement retry logic
- [ ] Set up logging and monitoring
- [ ] Configure cost alerts
- [ ] Test error recovery
- [ ] Validate output quality
- [ ] Document processing capacity
- [ ] Set up result backups
- [ ] Plan for scaling

## Next Steps

1. **Scale Up**: Test with larger document sets (100+)
2. **Optimize**: Profile and optimize bottlenecks
3. **Deploy**: Set up production batch processing jobs
4. **Monitor**: Implement comprehensive monitoring
5. **Iterate**: Continuously improve based on metrics

## Resources

- [Python concurrent.futures](https://docs.python.org/3/library/concurrent.futures.html)
- [NEUIToolkit README](../README.md)
- [Example 01: Getting Started](01_getting_started.ipynb)
- [Example 02: Neo4j Integration](02_neo4j_integration.ipynb)